# Distribution comparison results as one LaTeX table

Genera **un'unica tabella** con le due colonne iniziali **Model** e
**Representation**. Ogni modello occupa tre righe: **Base, MMFF, Atom details**.
Le colonne numeriche sono raggruppate per:

1. dataset: QM9, GDB13, ZINC (poi altri dataset);
2. split di riferimento: train, validation, test;
3. metriche: TVD complessiva e distanza Jensen–Shannon (JS).

Legge i JSON `outputs/distribution_comparison/<dataset>/<model>/<split>/details/scores/metrics.json`.
Ogni coppia dataset/split disponibile ha colonne TVD/JS condivise tra i modelli.
Le combinazioni mancanti per un modello vengono mostrate come `/`. Non si aggregano
modelli, dataset o split diversi. Le analisi marcate come incomplete o in corso
vengono escluse. I filtri permettono di restringere le colonne visualizzate.

Media e **SD campionaria (n−1)** provengono da `summary.macro_tvd` e
`summary.macro_js_distance`, sulla scala originale, senza percentuali.
**JS è una distanza**, cioè la radice della divergenza con logaritmi in base 2.
Non vengono ricalcolate le distanze e non si accede a W&B o ai modelli.

Resta attivo il requisito di **tre seed**: metriche con un conteggio diverso
sono mostrate come `/`, così come i valori mancanti. `SHOW_SEED_COUNTS = True`
aggiunge colonne `n` accanto alle metriche. Con `REQUIRED_SEEDS = None` si
mostrano tutti i conteggi; se le colonne `n` sono nascoste, il conteggio viene
indicato accanto ai valori. Con un solo seed la SD è non disponibile.

L'output visibile è stato generato dai JSON locali. Eseguire tutte le celle per
aggiornare la tabella dopo aver aggiunto nuovi risultati. `OUTPUT_DIR` permette
di esportare **un solo file**, `distribution_table.tex`; per impostazione
predefinita si stampa soltanto il LaTeX.

La tabella usa `table*` e viene adattata a `\textwidth` con `\resizebox`.
Nel preambolo LaTeX servono:

```latex
\usepackage{booktabs,graphicx,multirow}
\providecommand{\nlvalpm}[2]{\ensuremath{#1\,{\scriptstyle\pm}\,#2}}
```


In [ ]:
from pathlib import Path

# Run the notebook from the repository root, or set an absolute path here.
RESULTS_ROOT = Path("outputs/distribution_comparison")

# None selects everything available. For example: {"qm9", "gdb13"}, {"midi"}, {"test"}.
DATASET_FILTER = None
MODEL_FILTER = None
SPLIT_FILTER = None

REPRESENTATIONS = (("base", "Base"), ("mmff", "MMFF"), ("atom_details", "Atom details"))
METRICS = ("macro_tvd", "macro_js_distance")
DECIMALS = 6

# Keep the three-seed requirement; set to None to show any available seed count.
REQUIRED_SEEDS = 3

# Include separate seed-count columns (the compact default shows only TVD and JS).
SHOW_SEED_COUNTS = False

# Optional export of one combined .tex file; None only prints the table.
OUTPUT_DIR = None  # For example: Path("outputs/distribution_comparison/latex_tables").


In [ ]:
import json
import math
import sys


def normalize_split(value):
    if value is None:
        return "full"
    return {"valid": "validation", "val": "validation", "training": "train"}.get(value, value)


def normalize_dataset(value):
    return {"zinc": "zinc250k", "zinc_mmff": "zinc250k_mmff",
            "zinc_atom_details": "zinc250k_atom_details"}.get(value, value)


def scope_sort_key(scope):
    dataset, model, split = scope
    dataset_order = {"qm9": 0, "gdb13": 1, "zinc": 2, "zinc250k": 2}
    model_order = {"midi": 0, "d4": 1}
    split_order = {"train": 0, "validation": 1, "test": 2, "full": 3}
    return (dataset_order.get(dataset, 3), dataset,
            model_order.get(model, 2), model, split_order.get(split, 4), split)


def load_distribution_results(root):
    """Read the saved score summaries, keeping each dataset/model/split separate."""
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Distribution results directory not found: {root}. Check RESULTS_ROOT.")
    results = {}
    for path in sorted(root.glob("*/*/*/details/scores/metrics.json")):
        dataset, model, folder_split = path.relative_to(root).parts[:3]
        split = normalize_split(folder_split)
        if DATASET_FILTER is not None and dataset not in DATASET_FILTER:
            continue
        if MODEL_FILTER is not None and model not in MODEL_FILTER:
            continue
        if SPLIT_FILTER is not None and split not in {normalize_split(item) for item in SPLIT_FILTER}:
            continue
        report = path.parents[2] / "REPORT.md"
        if report.is_file():
            heading = report.read_text(encoding="utf-8").partition("\n")[0]
            if heading.startswith(("# Analisi non completata", "# Analisi in corso")):
                print(f"Skipping incomplete analysis: {report}", file=sys.stderr)
                continue
        document = json.loads(path.read_text(encoding="utf-8"))
        metadata = document["metadata"]
        identity = (normalize_dataset(metadata["reference_dataset"]), metadata["model"],
                    normalize_split(metadata["reference_split"]))
        scope = (dataset, model, split)
        if identity != (normalize_dataset(dataset), model, split):
            raise ValueError(f"Path/metadata mismatch in {path}: {scope} versus {identity}")
        if scope in results:
            raise ValueError(f"Duplicate result for {scope}: {path}")
        if not isinstance(document.get("representations"), dict):
            raise ValueError(f"Missing representation scores in {path}")
        results[scope] = {"path": path, "document": document}
    if not results:
        raise FileNotFoundError(f"No completed matching */*/*/details/scores/metrics.json under {root}")
    return dict(sorted(results.items(), key=lambda item: scope_sort_key(item[0])))


def latex_escape(text):
    replacements = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
                    "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
                    "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}
    return "".join(replacements.get(char, char) for char in str(text))


def metric_cell(entry, metric):
    """Use the saved mean/sample SD; never rescale or replace missing SD with zero."""
    stat = entry.get("summary", {}).get(metric, {})
    count = stat.get("n", 0)
    if not isinstance(count, int) or isinstance(count, bool) or count < 0:
        raise ValueError(f"Invalid number of seeds for {metric}: {count}")
    mean, std = stat.get("mean"), stat.get("std")
    if REQUIRED_SEEDS is not None and count != REQUIRED_SEEDS:
        return "/", str(count)
    if count == 0 or mean is None or not math.isfinite(mean):
        return "/", str(count)
    if std is None or count < 2 or not math.isfinite(std):
        return rf"\ensuremath{{{mean:.{DECIMALS}f}\;(\mathrm{{SD\ n/a}})}}", str(count)
    if std < 0:
        raise ValueError(f"Negative SD for {metric}: {std}")
    return rf"\nlvalpm{{{mean:.{DECIMALS}f}}}{{{std:.{DECIMALS}f}}}", str(count)


def make_distribution_table(results):
    """One table: model/representation rows, dataset/split/metric columns."""
    from itertools import groupby

    scopes = sorted(results, key=scope_sort_key)
    if not scopes:
        raise ValueError("No distribution results to tabulate")
    columns = sorted({(dataset, split) for dataset, _, split in scopes},
                     key=lambda column: scope_sort_key((column[0], "", column[1])))
    models = sorted({model for _, model, _ in scopes},
                    key=lambda model: ({"midi": 0, "d4": 1}.get(model, 2), model))
    width = len(METRICS) * (2 if SHOW_SEED_COUNTS else 1)
    labels = (
        {"qm9": "QM9", "gdb13": "GDB13", "zinc": "ZINC", "zinc250k": "ZINC"},
        {"train": "Train", "validation": "Validation", "test": "Test", "full": "Full"},
    )
    lines = ["% Sources:"]
    lines.extend("% " + results[scope]["path"].as_posix() for scope in scopes)
    lines.extend([
        r"\begin{table*}[t]",
        r"\setlength{\tabcolsep}{3pt}",
        r"\small",
        r"\centering",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{@{}ll" + "c" * (width * len(columns)) + r"@{}}",
        r"\toprule",
    ])
    # Each dataset/split gets one shared column group for every model row.
    for depth in range(2):
        headers, rules, start = [], [], 3
        for key, group in groupby(columns, key=lambda column: column[:depth + 1]):
            span = len(list(group)) * width
            label = latex_escape(labels[depth].get(key[-1], key[-1]))
            headers.append(rf"\multicolumn{{{span}}}{{c}}{{{label}}}")
            rules.append(rf"\cmidrule(lr){{{start}-{start + span - 1}}}")
            start += span
        lines.extend([" & & " + " & ".join(headers) + r" \\", " ".join(rules)])
    metric_labels = {"macro_tvd": r"TVD $\downarrow$", "macro_js_distance": r"JS $\downarrow$"}
    headers = []
    for column in columns:
        for metric in METRICS:
            headers.append(metric_labels[metric])
            if SHOW_SEED_COUNTS:
                headers.append("$n$")
    lines.extend(["Model & Representation & " + " & ".join(headers) + r" \\", r"\midrule"])
    for model_index, model in enumerate(models):
        if model_index:
            lines.append(r"\midrule")
        model_label = latex_escape({"midi": "MiDi", "d4": "D4"}.get(model, model))
        for row_index, (name, label) in enumerate(REPRESENTATIONS):
            model_cell = rf"\multirow{{{len(REPRESENTATIONS)}}}{{*}}{{{model_label}}}" if row_index == 0 else ""
            cells = [model_cell, latex_escape(label)]
            for dataset, split in columns:
                result = results.get((dataset, model, split), {})
                entry = result.get("document", {}).get("representations", {}).get(name, {})
                for metric in METRICS:
                    value, count = metric_cell(entry, metric)
                    if not SHOW_SEED_COUNTS and REQUIRED_SEEDS is None and value != "/":
                        value += rf" {{\scriptsize ($n={count}$)}}"
                    cells.append(value)
                    if SHOW_SEED_COUNTS:
                        cells.append(count)
            lines.append(" & ".join(cells) + r" \\")
    lines.extend([
        r"\bottomrule",
        r"\end{tabular}%",
        r"}",
        r"\caption{Overall distribution distances by dataset, model, and reference split. "
        r"Entries are mean $\pm$ sample SD across training seeds. "
        "JS denotes the Jensen--Shannon distance (square root of the divergence with base-2 logarithms). "
        "Values remain on their original scale; lower is better. "
        + (f"Only metrics with exactly {REQUIRED_SEEDS} seeds are shown. " if REQUIRED_SEEDS is not None else "")
        + (r"$n$ counts seeds with a defined score. " if SHOW_SEED_COUNTS or REQUIRED_SEEDS is None else "")
        + "Missing or excluded values are /; SD is undefined for a single seed.}",
        r"\label{tab:distribution-comparison}",
        r"\end{table*}",
    ])
    return "\n".join(lines)


In [ ]:
results = load_distribution_results(RESULTS_ROOT)
print(f"Loaded {len(results)} dataset/model/split combinations:")
for dataset, model, split in results:
    print(f"  {dataset} / {model} / {split}")


Loaded 6 dataset/model/split combinations:
  qm9 / midi / train
  qm9 / midi / validation
  qm9 / midi / test
  qm9 / d4 / train
  qm9 / d4 / validation
  qm9 / d4 / test


In [ ]:
latex_output = make_distribution_table(results)
print(latex_output)

if OUTPUT_DIR is not None:
    output = Path(OUTPUT_DIR) / "distribution_table.tex"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(latex_output + "\n", encoding="utf-8")


% Sources:
% /mnt/projects/dpg/hit/u0074/repository/D4-distance-diffusion/outputs/distribution_comparison/qm9/midi/train/details/scores/metrics.json
% /mnt/projects/dpg/hit/u0074/repository/D4-distance-diffusion/outputs/distribution_comparison/qm9/midi/validation/details/scores/metrics.json
% /mnt/projects/dpg/hit/u0074/repository/D4-distance-diffusion/outputs/distribution_comparison/qm9/midi/test/details/scores/metrics.json
% /mnt/projects/dpg/hit/u0074/repository/D4-distance-diffusion/outputs/distribution_comparison/qm9/d4/train/details/scores/metrics.json
% /mnt/projects/dpg/hit/u0074/repository/D4-distance-diffusion/outputs/distribution_comparison/qm9/d4/validation/details/scores/metrics.json
% /mnt/projects/dpg/hit/u0074/repository/D4-distance-diffusion/outputs/distribution_comparison/qm9/d4/test/details/scores/metrics.json
\begin{table*}[t]
\setlength{\tabcolsep}{3pt}
\small
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{@{}llcccccc@{}}
\toprule
 & & \multicolumn{6}{c}{QM